# Análisis de tasa de reconocimiento — SHHH!

Este notebook procesa trazas `.json` de sesiones del proyecto **SHHH!** y genera un CSV con una fila por sesión.

Cada archivo `.json` debe representar una sesión. El nombre del archivo se usa como identificador del participante/sesión, por ejemplo:

- `Esther0.json`
- `Esther1.json`
- `Juan0.json`

El campo `app_version` dentro de los eventos debe indicar la versión probada:

- `1.0-base`
- `1.1-semantic`
- `1.1-contextual`

La métrica calculada es:

```text
Tasa de reconocimiento = comandos reconocidos / (comandos reconocidos + comandos no reconocidos) * 100
```

Eventos usados:

- `voice_command_recognized`
- `voice_command_not_recognized`


In [1]:
from pathlib import Path
import json
import re
import pandas as pd

# Carpeta donde están los JSON de las sesiones.
# Por defecto, el notebook buscará JSON en la misma carpeta donde se ejecute.
INPUT_FOLDER = Path(".")

# Nombre del CSV de salida.
OUTPUT_CSV = Path("resultados_tasa_reconocimiento.csv")

# Eventos usados para calcular la métrica.
RECOGNIZED_EVENT = "voice_command_recognized"
NOT_RECOGNIZED_EVENT = "voice_command_not_recognized"

# Versiones esperadas. No es obligatorio que aparezcan todas.
EXPECTED_VERSIONS = {"1.0-base", "1.1-semantic", "1.1-contextual"}


In [11]:
def load_events_from_json_file(path: Path):
    """
    Carga eventos desde un archivo JSON.

    Soporta dos formatos:
    1. JSON Lines: un objeto JSON por línea.
    2. JSON normal: una lista de objetos o un único objeto.
    """
    text = path.read_text(encoding="utf-8-sig").strip()

    if not text:
        return []

    # Formato JSON Lines: una entrada por línea.
    events = []
    json_lines_ok = True

    for line_number, line in enumerate(text.splitlines(), start=1):
        line = line.strip()
        if not line:
            continue

        try:
            events.append(json.loads(line))
        except json.JSONDecodeError:
            json_lines_ok = False
            break

    if json_lines_ok and events:
        return events

    # Formato JSON estándar.
    data = json.loads(text)

    if isinstance(data, list):
        return data

    if isinstance(data, dict):
        return [data]

    raise ValueError(f"Formato JSON no soportado en {path.name}")


def parse_session_name(file_stem: str):
    """
    Extrae el nombre del participante y el índice de sesión a partir del nombre del archivo.

    Ejemplos:
    Esther0 -> participant = Esther, session_index = 0
    Juan1   -> participant = Juan, session_index = 1
    """
    match = re.match(r"^(.*?)(\d+)$", file_stem)

    if match:
        participant = match.group(1)
        session_index = int(match.group(2))
    else:
        participant = file_stem
        session_index = None

    return participant, session_index


def get_app_version(events):
    """
    Obtiene app_version de la sesión.

    Si hay varias versiones en el mismo archivo, se devuelve la más frecuente.
    """
    versions = [
        event.get("app_version")
        for event in events
        if isinstance(event, dict) and event.get("app_version")
    ]

    if not versions:
        return "unknown"

    return pd.Series(versions).mode().iloc[0]


def calculate_session_metrics(json_path: Path):
    events = load_events_from_json_file(json_path)

    participant, session_index = parse_session_name(json_path.stem)
    app_version = get_app_version(events)

    recognized_count = 0
    not_recognized_count = 0

    for event in events:
        if not isinstance(event, dict):
            continue

        event_type = event.get("event_type")

        if event_type == RECOGNIZED_EVENT:
            recognized_count += 1
        elif event_type == NOT_RECOGNIZED_EVENT:
            not_recognized_count += 1

    total_attempts = recognized_count + not_recognized_count

    if total_attempts > 0:
        recognition_rate = round(recognized_count / total_attempts, 4)
        recognition_rate_percent = round(recognition_rate * 100, 4)
    else:
        recognition_rate = None
        recognition_rate_percent = None

    return {
        "file_name": json_path.name,
        "participant": participant,
        "session_index": session_index,
        "app_version": app_version,
        "recognized_commands": recognized_count,
        "not_recognized_commands": not_recognized_count,
        "total_voice_attempts": total_attempts,
        "recognition_rate": recognition_rate,
        "recognition_rate_percent": recognition_rate_percent,
    }


In [15]:
json_files = sorted(INPUT_FOLDER.glob("*.json"))

if not json_files:
    raise FileNotFoundError(
        f"No se encontraron archivos .json en la carpeta: {INPUT_FOLDER.resolve()}"
    )

rows = []

for json_file in json_files:
    try:
        rows.append(calculate_session_metrics(json_file))
    except Exception as error:
        rows.append({
            "file_name": json_file.name,
            "participant": json_file.stem,
            "session_index": None,
            "app_version": "error",
            "recognized_commands": None,
            "not_recognized_commands": None,
            "total_voice_attempts": None,
            "recognition_rate": None,
            "recognition_rate_percent": None,
            "error": str(error),
        })

results_df = pd.DataFrame(rows)

# Orden recomendado de columnas.
columns = [
    "file_name",
    "participant",
    "session_index",
    "app_version",
    "recognized_commands",
    "not_recognized_commands",
    "total_voice_attempts",
    "recognition_rate",
    "recognition_rate_percent",
]

# Si hubo errores, mantenemos también la columna error.
if "error" in results_df.columns:
    columns.append("error")

results_df = results_df[columns]

results_df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")

print(f"CSV generado correctamente: {OUTPUT_CSV.resolve()}")
display(results_df)


CSV generado correctamente: C:\Users\PC\Documents\GitHub\UAJ_Final_Project\Analysis\resultados_tasa_reconocimiento.csv


,file_name,participant,session_index,app_version,recognized_commands,not_recognized_commands,total_voice_attempts,recognition_rate,recognition_rate_percent
0,Esther0.json,Esther,0,1.1-contextual,81,172,253,0.3202,32.02


## Interpretación

La columna principal para comparar versiones es:

```text
recognition_rate_percent
```

Para cada versión:

- `1.0-base`: versión sin mejoras.
- `1.1-semantic`: versión con flexibilidad semántica.
- `1.1-contextual`: versión con comandos contextuales.

Una tasa más alta indica que, durante esa sesión, una mayor proporción de intentos de voz fueron reconocidos como comandos válidos.
